# Real-Time Clothing Segmentation & Virtual Try-On (Colab -> ONNX -> Rust Mobile)

Notebook này dùng để **train & fine-tune** mô hình nhận diện/tách nền quần áo (`YOLO11n-seg`) trên Google Colab, sau đó xuất ra file `.onnx` tối ưu tốc độ cho **Rust mobile engine** (chạy real-time trên camera điện thoại).

In [ ]:
# 1. Cài đặt thư viện (Bật GPU T4 trên Colab: Runtime -> Change runtime type -> T4 GPU)
!pip install -q ultralytics onnx onnxsim onnxruntime roboflow

In [ ]:
# 2. Kiểm tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Chuẩn bị Dataset (Chọn 1 trong 2 cách)
# --- CÁCH A: Tải dataset mẫu từ Roboflow (thay YOUR_API_KEY bằng key miễn phí của bạn) ---
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("fashion-ai").project("clothing-segmentation")
# dataset = project.version(1).download("yolov11")
# DATA_YAML = f"{dataset.location}/data.yaml"

# --- CÁCH B: Mount Google Drive chứa dataset tự gán nhãn (YOLO Segmentation format) ---
from google.colab import drive
drive.mount('/content/drive')

# Tạo file data.yaml mẫu (chỉnh lại đường dẫn thư mục ảnh của bạn trên Drive)
data_yaml_content = """
path: /content/drive/MyDrive/clothes_dataset
train: images/train
val: images/val

names:
  0: shirt
  1: pants
  2: jacket
  3: dress
"""
with open("data.yaml", "w") as f:
    f.write(data_yaml_content.strip())
DATA_YAML = "data.yaml"

# Kiểm tra nhanh số lượng ảnh (Nam & Nữ) và số đỉnh polygon thực tế trong labels/
from pathlib import Path
ds_root = Path("/content/drive/MyDrive/clothes_dataset")
if ds_root.exists():
    for sp in ("train", "val"):
        imgs = list((ds_root / "images" / sp).glob("*.jpg"))
        lbls = list((ds_root / "labels" / sp).glob("*.txt"))
        pts_counts = [ (len(p.read_text().strip().split()) - 1) // 2 for p in lbls if p.read_text().strip() ]
        print(f"{sp}: {len(imgs)} images, {len(lbls)} labels, polygon vertices min={min(pts_counts, default=0)}, max={max(pts_counts, default=0)}")

In [ ]:
# 4. Train & Fine-tune YOLO11n-seg (bản Nano tối ưu cho real-time trên điện thoại)
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")  # Tải pre-trained weights

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=320,          # 320x320 giúp Rust chạy >= 25-30 FPS trên mobile
    batch=16,
    patience=10,        # Early stopping nếu không cải thiện
    device=0,
    hsv_h=0.015,        # Augmentation ánh sáng camera điện thoại
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,
    project="clothes_tryon",
    name="yolo11n_seg_clothes"
)

In [ ]:
# 5. Đánh giá mô hình sau khi fine-tune
best_model = YOLO("clothes_tryon/yolo11n_seg_clothes/weights/best.pt")
metrics = best_model.val()
print(f"mAP50 (Mask): {metrics.seg.map50:.4f}")

In [ ]:
# 6. Xuất sang ONNX tối ưu cho Rust (ort crate) và tải về máy
onnx_path = best_model.export(
    format="onnx",
    imgsz=320,
    simplify=True,
    opset=17,
    dynamic=False
)
print("Exported ONNX FP32 model:", onnx_path)

# (Tùy chọn) Lượng tử hóa INT8 để giảm ~3.5x dung lượng cho mobile/WASM
from onnxruntime.quantization import QuantType, quantize_dynamic
int8_path = str(onnx_path).replace(".onnx", "_int8.onnx")
quantize_dynamic(model_input=str(onnx_path), model_output=int8_path, weight_type=QuantType.QUInt8)
print("Exported ONNX INT8 model:", int8_path)

from google.colab import files
files.download(onnx_path)